## Multi-Agent System with Google ADK

Building on the Challenge 2 weather agent, this notebook creates a
multi-agent system with:

- **Root Agent** — receives user requests and delegates to the appropriate sub-agent
- **Weather Agent** — provides US weather forecasts (from Challenge 2, with callbacks)
- **Search Agent** — answers general questions using Google Search

The root agent inspects each request and routes it to whichever
sub-agent is best suited to handle it.

## 1. Setup & Dependencies

In [1]:
!pip install google-adk -q
!pip install litellm -q
!pip install google-cloud-modelarmor -q

In [2]:
import os
import re
import requests
import time
from typing import Optional, Tuple, Dict

import google.auth
from google.genai import types
from google import genai
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse

from google.cloud import modelarmor_v1
from google.api_core.client_options import ClientOptions

## 2. Configuration

In [3]:
project = google.auth.default()[1]

GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "GOOGLE_MAPS_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = project
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

LOCATION = "us-central1"
MODEL_ARMOR_TEMPLATE = "MODEL_ARMOR_TEMPLATE_NAME"

# Model Armor client
armor_client = modelarmor_v1.ModelArmorClient(
    transport="rest",
    client_options=ClientOptions(
        api_endpoint=f"modelarmor.{LOCATION}.rep.googleapis.com"
    ),
)

# Vertex AI GenAI client (used by the custom search tool)
genai_client = genai.Client(vertexai=True, project=project, location=LOCATION)

## 3. Weather Tool Functions

Carried forward from Challenge 2.

Deviating from the instructor's code snippet a bit to test another method for accomplishing the search piece (preserving ADK's native multi-agent delegation pattern).  Adding a sub-agent with a Google search tool rather than giving the tool directly to root.


```
config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
        )
```
instead of


```
tools=[agent_tool.AgentTool(agent=google_search_agent)],
```




In [4]:
def geocode_location(place: str, api_key: str) -> Tuple[float, float]:
    """Convert a place name into latitude and longitude."""
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": place, "key": api_key}

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    if not data.get("results"):
        raise ValueError(f"No geocoding results found for '{place}'.")

    location = data["results"][0]["geometry"]["location"]
    return location["lat"], location["lng"]


def get_weather_forecast(latitude: float, longitude: float) -> Dict:
    """Retrieve the current weather forecast from the National Weather Service."""
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
    points_response = requests.get(points_url)
    points_response.raise_for_status()
    points_data = points_response.json()

    forecast_url = points_data["properties"]["forecast"]
    forecast_response = requests.get(forecast_url)
    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()

    return forecast_data["properties"]["periods"][0]


def weather_lookup(city: str) -> str:
    """Retrieve a weather summary for a US city.

    Args:
        city: A US city name, ideally with state (e.g., "Denver, CO").

    Returns:
        A string containing the detailed weather forecast.
    """
    try:
        time.sleep(2)  # Avoid rate limiting on the Geocoding API
        lat, lon = geocode_location(city, GOOGLE_MAPS_API_KEY)
        forecast = get_weather_forecast(lat, lon)
        return forecast["detailedForecast"]
    except (ValueError, KeyError, requests.RequestException) as e:
        return f"Unable to retrieve weather for '{city}': {e}"


def search_the_web(query: str) -> str:
    """Search the web for information using Google Search.

    Use this tool to find up-to-date information about any topic.

    Args:
        query: The search query to look up.

    Returns:
        A text summary of the search results.
    """
    response = genai_client.models.generate_content(
        model="gemini-2.0-flash",
        contents=query,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
        ),
    )
    return response.text

## 4. Validation Helpers

In [5]:
def extract_location(text: str) -> Optional[str]:
    """Extract a location name from a user prompt."""
    match = re.search(
        r'\b(?:in|for|at|of)\s+(.+?)(?:\?|$|\.|!)',
        text,
        re.IGNORECASE,
    )
    if match:
        return match.group(1).strip()
    return None


def is_us_location(location: str) -> bool:
    """Check whether a location string geocodes to the United States."""
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": location, "key": GOOGLE_MAPS_API_KEY}
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException:
        return True

    if not data.get("results"):
        return True

    for component in data["results"][0].get("address_components", []):
        if "country" in component.get("types", []):
            return component.get("short_name") == "US"

    return True


def check_model_armor(text: str) -> Tuple[bool, str]:
    """Run text through Model Armor for content safety."""
    template_path = (
        f"projects/{project}/locations/{LOCATION}"
        f"/templates/{MODEL_ARMOR_TEMPLATE}"
    )
    request = modelarmor_v1.SanitizeUserPromptRequest(
        name=template_path,
        user_prompt_data=modelarmor_v1.DataItem(text=text),
    )
    response = armor_client.sanitize_user_prompt(request=request)
    match_state = response.sanitization_result.filter_match_state

    if match_state == modelarmor_v1.FilterMatchState.MATCH_FOUND:
        details = []
        for name, result in response.sanitization_result.filter_results.items():
            details.append(name)
        reason = f"Content flagged by Model Armor ({', '.join(details)})."
        return False, reason

    return True, ""

## 5. Callback Functions

Carried forward from Challenge 2 — logging and input moderation.

In [6]:
def _get_latest_user_text(llm_request: LlmRequest) -> Optional[str]:
    """Extract the most recent user text from an LlmRequest."""
    if not llm_request.contents:
        return None
    for content in reversed(llm_request.contents):
        if content.role != "user" or not content.parts:
            continue
        if any(getattr(p, "function_response", None) for p in content.parts):
            continue
        for part in content.parts:
            if getattr(part, "text", None):
                return part.text
    return None


def log_user_prompt(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Log user prompts before they are sent to the model."""
    user_text = _get_latest_user_text(llm_request)
    if user_text:
        print(
            f"[LOG] User prompt → {callback_context.agent_name}: "
            f"{user_text}"
        )
    return None


def log_model_response(
    callback_context: CallbackContext, llm_response: LlmResponse
) -> Optional[LlmResponse]:
    """Log model responses after generation."""
    if llm_response.content and llm_response.content.parts:
        for part in llm_response.content.parts:
            if getattr(part, "text", None):
                preview = part.text[:200]
                print(
                    f"[LOG] Model response ← {callback_context.agent_name}: "
                    f"{preview}{'...' if len(part.text) > 200 else ''}"
                )
            elif getattr(part, "function_call", None):
                print(
                    f"[LOG] Tool call ← {callback_context.agent_name}: "
                    f"{part.function_call.name}()"
                )
    return None


def moderate_user_prompt(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Validate user input — blocks non-US locations & unsafe content."""
    user_text = _get_latest_user_text(llm_request)
    if not user_text:
        return None

    is_safe, reason = check_model_armor(user_text)
    if not is_safe:
        print(f"[BLOCKED] {reason}")
        return LlmResponse(
            content=types.Content(
                role="model",
                parts=[types.Part(text=f"I cannot process this request. {reason}")],
            )
        )

    location = extract_location(user_text)
    if location and not is_us_location(location):
        print(f"[BLOCKED] Non-US location detected: {location}")
        return LlmResponse(
            content=types.Content(
                role="model",
                parts=[
                    types.Part(
                        text=(
                            "I can only provide weather forecasts for US locations. "
                            "The National Weather Service API only covers the "
                            "United States."
                        )
                    )
                ],
            )
        )

    return None

## 6. Agent Definitions

Three agents form the multi-agent system:

| Agent | Role | Tools |
|---|---|---|
| `weather_agent` | Fetches US weather forecasts | `weather_lookup` |
| `search_agent` | Answers general knowledge questions | `google_search` |
| `root_agent` | Routes requests to the right sub-agent | *(delegates)* |

In [7]:
# --- Weather Agent (from Challenge 2) ---
weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.0-flash",
    description=(
        "Provides current weather forecasts for US cities. "
        "Use this agent when the user asks about weather, "
        "temperature, or forecast conditions in a US location."
    ),
    instruction="""
You are a weather assistant for US cities.

When a user asks about the weather in a city, use the weather_lookup
tool to retrieve the current forecast. Then provide:
  1. A clear, concise summary of current conditions.
  2. The temperature and any notable weather alerts.
  3. A brief recommendation (e.g., bring an umbrella, wear layers).

If the tool returns an error, let the user know and suggest they
check the city name or try again.
""",
    tools=[weather_lookup],
    before_model_callback=[log_user_prompt, moderate_user_prompt],
    after_model_callback=log_model_response,
)

In [8]:
# --- Search Agent (new for Challenge 3) ---
# Note: We use a custom search_the_web wrapper instead of the built-in
# google_search tool. The built-in tool is a Gemini "grounding" tool that
# cannot be mixed with regular function tools (like the transfer tools ADK
# adds automatically in multi-agent setups), causing an INVALID_ARGUMENT error.
search_agent = Agent(
    name="search_agent",
    model="gemini-2.0-flash",
    description=(
        "Answers general knowledge questions by searching the web. "
        "Use this agent for any question that is NOT about weather "
        "or forecasts — e.g., news, facts, how-to questions, sports, "
        "history, science, or current events."
    ),
    instruction="""
You are a research assistant with access to Google Search.

When a user asks a question, use the search_the_web tool to find
up-to-date information. Then provide:
  1. A clear, concise answer based on the search results.
  2. Key facts or data points that support the answer.
  3. If relevant, mention the source of the information.

Always ground your answers in the search results rather than
relying on prior knowledge alone.
""",
    tools=[search_the_web],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

In [9]:
# --- Root Agent (orchestrator) ---
root_agent = Agent(
    name="root_agent",
    model="gemini-2.0-flash",
    description="The main coordinating agent that routes user requests.",
    instruction="""
You are the main coordinating agent. Your job is to understand what
the user needs and delegate to the right specialist:

- **weather_agent**: For any questions about weather, temperature,
  forecasts, or climate conditions in US cities.
- **search_agent**: For general knowledge questions, current events,
  facts, how-to questions, or anything that is not weather-related.

Always delegate to a sub-agent — do not try to answer questions
yourself. If the user's intent is unclear, ask a brief clarifying
question.
""",
    sub_agents=[weather_agent, search_agent],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

## 7. Test Harness

Run the root agent against various prompts and print all events
to demonstrate sub-agent delegation.

In [10]:
async def test_multi_agent(prompts: list[str]) -> None:
    """Run the root agent against prompts, printing events to show delegation."""
    session_service = InMemorySessionService()
    runner = Runner(
        agent=root_agent,
        app_name="multi_agent_system",
        session_service=session_service,
    )

    for i, prompt in enumerate(prompts):
        session_id = f"test_session_{i}"

        await session_service.create_session(
            app_name="multi_agent_system",
            user_id="test_user",
            session_id=session_id,
        )

        content = types.Content(
            role="user",
            parts=[types.Part(text=prompt)],
        )

        print(f"\n{'=' * 60}")
        print(f"User: {prompt}")
        print(f"{'=' * 60}")

        async for event in runner.run_async(
            user_id="test_user",
            session_id=session_id,
            new_message=content,
        ):
            # Print every event to show which agent is handling what
            agent_name = getattr(event, "author", "unknown")
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if getattr(part, "text", None):
                        print(f"  [{agent_name}] {part.text[:300]}")
                    elif getattr(part, "function_call", None):
                        print(f"  [{agent_name}] calling {part.function_call.name}()")
                    elif getattr(part, "function_response", None):
                        print(f"  [{agent_name}] tool result received")

            if event.is_final_response():
                final_text = event.content.parts[0].text
                print(f"\n  >> Final answer ({agent_name}): {final_text}\n")

### 7a. Weather Requests — Routed to `weather_agent`

In [11]:
weather_prompts = [
    "What is the weather in Chicago, IL?",
    "What's the forecast for Denver, CO?",
]

await test_multi_agent(weather_prompts)


User: What is the weather in Chicago, IL?
[LOG] User prompt → root_agent: What is the weather in Chicago, IL?


[LOG] Tool call ← root_agent: transfer_to_agent()
  [root_agent] calling transfer_to_agent()
  [root_agent] tool result received
[LOG] User prompt → weather_agent: For context:
[LOG] Tool call ← weather_agent: weather_lookup()
  [weather_agent] calling weather_lookup()
  [weather_agent] tool result received
[LOG] User prompt → weather_agent: For context:
[LOG] Model response ← weather_agent: The weather in Chicago, IL is cloudy with areas of fog and a slight chance of drizzle. The high will be near 39 degrees with a north wind around 5 mph. There is a 20% chance of precipitation, with new...
  [weather_agent] The weather in Chicago, IL is cloudy with areas of fog and a slight chance of drizzle. The high will be near 39 degrees with a north wind around 5 mph. There is a 20% chance of precipitation, with new rainfall amounts less than a tenth of an inch possible. Consider taking extra caution while driving

  >> Final answer (weather_agent): The weather in Chicago, IL is cloudy with area

### 7b. Search Requests — Routed to `search_agent`

In [12]:
search_prompts = [
    "Who won the most recent Super Bowl?",
    "What is the capital of France?",
]

await test_multi_agent(search_prompts)


User: Who won the most recent Super Bowl?
[LOG] User prompt → root_agent: Who won the most recent Super Bowl?
[LOG] Tool call ← root_agent: transfer_to_agent()
  [root_agent] calling transfer_to_agent()
  [root_agent] tool result received
[LOG] User prompt → search_agent: For context:
[LOG] Tool call ← search_agent: search_the_web()
  [search_agent] calling search_the_web()
  [search_agent] tool result received
[LOG] User prompt → search_agent: For context:
[LOG] Model response ← search_agent: The Philadelphia Eagles won Super Bowl LIX (2025), defeating the Kansas City Chiefs 40-22. The game was played on February 9, 2025, at the Caesars Superdome in New Orleans. Jalen Hurts was named Super...
  [search_agent] The Philadelphia Eagles won Super Bowl LIX (2025), defeating the Kansas City Chiefs 40-22. The game was played on February 9, 2025, at the Caesars Superdome in New Orleans. Jalen Hurts was named Super Bowl MVP.


  >> Final answer (search_agent): The Philadelphia Eagles won Supe

### 7c. Mixed Requests — Demonstrates Routing Logic

A mix of weather and general queries to show the root agent
correctly delegating each to the right sub-agent.

In [13]:
mixed_prompts = [
    "What is the weather in Miami, FL?",
    "How tall is the Eiffel Tower?",
    "What's the temperature in Seattle, WA?",
    "What year did the Titanic sink?",
]

await test_multi_agent(mixed_prompts)


User: What is the weather in Miami, FL?
[LOG] User prompt → root_agent: What is the weather in Miami, FL?
[LOG] Tool call ← root_agent: transfer_to_agent()
  [root_agent] calling transfer_to_agent()
  [root_agent] tool result received
[LOG] User prompt → weather_agent: For context:
[LOG] Tool call ← weather_agent: weather_lookup()
  [weather_agent] calling weather_lookup()
  [weather_agent] tool result received
[LOG] User prompt → weather_agent: For context:
[LOG] Model response ← weather_agent: OK. The weather in Miami, FL is mostly sunny with a high near 80 degrees. There is a slight chance of rain showers after 3pm. The wind will be around 14 mph, with gusts as high as 21 mph. The chance o...
  [weather_agent] OK. The weather in Miami, FL is mostly sunny with a high near 80 degrees. There is a slight chance of rain showers after 3pm. The wind will be around 14 mph, with gusts as high as 21 mph. The chance of precipitation is 20%. It would be a good idea to bring an umbrella just in

### 7d. Validation Still Works — Non-US Weather Blocked

The weather agent's `moderate_user_prompt` callback still blocks
non-US locations even when requests arrive through the root agent.

In [14]:
validation_prompts = [
    "What is the weather in London, England?",
    "What is the weather in Tokyo, Japan?",
]

await test_multi_agent(validation_prompts)


User: What is the weather in London, England?
[LOG] User prompt → root_agent: What is the weather in London, England?
[LOG] Model response ← root_agent: Since you're asking about the weather, but not in a US city, I will transfer you to the search agent.

[LOG] Tool call ← root_agent: transfer_to_agent()
  [root_agent] Since you're asking about the weather, but not in a US city, I will transfer you to the search agent.

  [root_agent] calling transfer_to_agent()
  [root_agent] tool result received
[LOG] User prompt → search_agent: For context:
[LOG] Tool call ← search_agent: search_the_web()
  [search_agent] calling search_the_web()
  [search_agent] tool result received
[LOG] User prompt → search_agent: For context:
[LOG] Model response ← search_agent: The weather in London, England is currently cloudy with a temperature of 52°F (11°C), but feels like 52°F (11°C). There is a 10% chance of rain. The forecast for the next few days is:

*   **Tonight:*...
  [search_agent] The weather in 